# 06 — A demanding case: breast cancer

*Notebook 6 of 6 — Logistic Regression*

This is where the chapter comes together. We take logistic regression to a real diagnostic problem —
breast-cancer tumours — and run the honest workflow end to end. Two things that did not matter on
penguins matter a great deal here: **can we trust the probability**, and **where do we set the
threshold** when missing a cancer is far worse than a false alarm?

**Prerequisites.** NB 1–5 (the whole method, by hand and with the estimator). Module 00: confusion /
precision / recall (NB 07), score → threshold and ROC/PR (NB 08), cross-validation (NB 10), the
`Pipeline` and fit-on-train (NB 11). Chapter 01 NB 5 (KNN on this same dataset). Chapter 02 NB 5
(calibration, the reliability diagram, the Brier score, and naive Bayes' over-confidence).

**What you'll be able to do.**
- Run an honest logistic-regression workflow: split, cross-validate on train, read the sealed test once.
- Read a **reliability diagram** and decide whether to trust a probability.
- Choose a **decision threshold** under an asymmetric cost.
- Read the **coefficients**, and let **L1** select the features that matter.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course import colors, datasets, viz

viz.use_course_style()
np.random.seed(0)

df = datasets.load_breast_cancer()
X = df.drop(columns="target")
# malignant = 1 = positive (the costly miss). sklearn encodes malignant as target 0, so we flip.
y = (df["target"] == 0).astype(int)
diagnosis = y.map({1: "malignant", 0: "benign"})
print(f"{df.shape[0]} patients, {X.shape[1]} features | "
      f"malignant {int(y.sum())} / benign {int((y == 0).sum())}")

## Where we are, and the stakes

569 patients, 30 measurements taken from each tumour's cells, **212 malignant and 357 benign**. This is
the dataset k-nearest neighbours met in chapter 01 — where it *felt* the curse of dimensionality. A
linear model will read it cleanly, and, crucially, report **probabilities we can check**.

We treat **malignant as the positive class** — the case we must not miss. Everything is standardized
inside a `Pipeline` and fitted on the training data only, so no information leaks from the test set.

In [ ]:
fig = viz.plot_class_balance(diagnosis)
fig.axes[0].set_title("Diagnosis balance (malignant = the positive class)")
plt.show()

df[["mean radius", "mean concave points", "worst area"]].describe().round(2)

**Read the figure.** About **37%** of the tumours are malignant — the classes are uneven. That means
**accuracy alone can mislead**: a model that called everything "benign" would already score 63%. So we
will watch **recall on malignant** (how many cancers we catch) and **calibration** (whether the
probabilities are trustworthy), not accuracy by itself.

## The honest workflow

Split once. **Cross-validate on the training set** to compare two models — logistic regression and the
naive Bayes of chapter 02 — then read the **sealed test set exactly once** (module 00, NB 10). Both
models live inside one standardized `Pipeline`, fitted on train only.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=10000))
gnb = make_pipeline(StandardScaler(), GaussianNB())

print("5-fold CV accuracy on the training set:")
print(f"  LogisticRegression : {cross_val_score(logreg, Xtr, ytr, cv=cv).mean():.4f}")
print(f"  GaussianNB         : {cross_val_score(gnb, Xtr, ytr, cv=cv).mean():.4f}")

logreg.fit(Xtr, ytr)
gnb.fit(Xtr, ytr)
print(f"\nheld-out test accuracy:  LogReg {logreg.score(Xte, yte):.4f}  |  "
      f"GaussianNB {gnb.score(Xte, yte):.4f}")

**Read the result.** Logistic regression wins cross-validation on the training set by about five points
(0.985 vs 0.932) and scores 0.953 on the sealed test. But on a medical problem, accuracy is not the whole
story. The next question is whether the *probabilities* it reports can be trusted.

## Calibration: closing the chapter-02 loop

A **calibrated** model is one whose "0.9" really does come true about 90% of the time. In chapter 02 we
saw naive Bayes was **over-confident** — it piled its probabilities at 0 and 1, double-counting
correlated evidence. Logistic regression models $P(y \mid x)$ directly, so its probabilities should be
better behaved. Let us check, both standardized, on the same test set: a **reliability diagram** (observed
frequency vs predicted probability) and the **Brier score** (mean squared error of the probabilities,
lower is better).

In [ ]:
p_lr = logreg.predict_proba(Xte)[:, 1]
p_nb = gnb.predict_proba(Xte)[:, 1]
brier_lr, brier_nb = brier_score_loss(yte, p_lr), brier_score_loss(yte, p_nb)
pile_lr = int(np.sum((p_lr > 0.99) | (p_lr < 0.01)))
pile_nb = int(np.sum((p_nb > 0.99) | (p_nb < 0.01)))
print(f"Brier score:  LogReg {brier_lr:.4f}  |  GaussianNB {brier_nb:.4f}")
print(f"piled past 0.99 / 0.01:  LogReg {pile_lr}/{len(yte)}  |  GaussianNB {pile_nb}/{len(yte)}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 5))
axL.plot([0, 1], [0, 1], color=colors.COLORS["muted"], linestyle="--", linewidth=1, label="perfect")
viz.plot_calibration_curve(yte, p_lr, label="LogReg", color=colors.COLORS["model"], ax=axL)
viz.plot_calibration_curve(yte, p_nb, label="GaussianNB", color=colors.COLORS["error"], ax=axL)
axL.set_title(f"Reliability (Brier: LogReg {brier_lr:.3f}, NB {brier_nb:.3f})")
bins = np.linspace(0, 1, 21)
axR.hist(p_lr, bins=bins, alpha=0.6, color=colors.COLORS["model"], label="LogReg")
axR.hist(p_nb, bins=bins, alpha=0.6, color=colors.COLORS["error"], label="GaussianNB")
axR.set_xlabel("predicted P(malignant)")
axR.set_ylabel("count")
axR.set_title("Where the probabilities pile up")
axR.legend()
plt.tight_layout()
plt.show()

**Read the figure.** *Left:* in shape both models roughly track the diagonal — neither is wildly
miscalibrated — though GaussianNB drifts off it at the top (it calls cases "1.0" that are malignant only
~90% of the time). *Right:* the telling difference is **where the probabilities sit** — GaussianNB crowds
them hard against 0 and 1 (**166 of 171** at the extremes) far more than LogReg (**119**). That is
**over-confidence**, exactly as chapter 02 warned, and it costs when the model is confidently wrong: the
**Brier score is ~3× worse** (0.098 vs 0.033). Honest nuance: near-separable data still pushes LogReg
fairly confident too — **better-calibrated, not perfect**; recalibrate (`CalibratedClassifierCV`) if the
number must be exact.

## The threshold is a policy, not a default

Predicting "malignant" when $P \ge 0.5$ is only the default cut. A **missed cancer is far costlier than a
false alarm**: a false alarm leads to a follow-up; a missed cancer can be fatal. So we should be willing
to flag more aggressively — **lower the threshold** — and accept more false alarms. Because the
probability is calibrated, this is a **defensible policy**, not a guess (module 00, NB 08, made real).

In [ ]:
thresholds = np.linspace(0.02, 0.98, 49)
recalls = [recall_score(yte, p_lr >= t) for t in thresholds]
precisions = [precision_score(yte, p_lr >= t, zero_division=0) for t in thresholds]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))
viz.plot_score_threshold(p_lr, yte, threshold=0.5, class_names=("benign", "malignant"), ax=axL)
axL.set_title("Predicted P(malignant): the classes overlap")
axR.plot(thresholds, recalls, color=colors.COLORS["model"], linewidth=2, label="recall (malignant)")
axR.plot(thresholds, precisions, color=colors.COLORS["highlight"], linewidth=2, label="precision")
axR.axvline(0.5, color=colors.COLORS["grid"], linewidth=1, linestyle="--")
axR.set_xlabel("threshold on P(malignant)")
axR.set_ylabel("score")
axR.set_title("Recall and precision as the threshold moves")
axR.legend(loc="lower center")
plt.tight_layout()
plt.show()

**Read the figure.** *Left:* the two classes' predicted probabilities overlap in a middle band; the
dashed line is the 0.5 cut. *Right:* slide the threshold **left** (flag more) and **recall climbs** (we
catch more cancers) while **precision falls** (more benign tumours flagged). That trade is the decision —
and it is ours to set, with the cost of each mistake in mind.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.8))
for thr, ax in [(0.5, axL), (0.1, axR)]:
    pred = (p_lr >= thr).astype(int)
    cm = confusion_matrix(yte, pred)  # rows = true [benign, malignant], cols = predicted
    viz.plot_confusion_matrix(cm, class_names=["benign", "malignant"], ax=ax)
    ax.set_title(f"thr {thr}: recall {recall_score(yte, pred):.3f}, {cm[1, 0]} missed")
plt.tight_layout()
plt.show()

**Read the figure.** At the default **0.5** the model misses **4 of 64 cancers** (4 false alarms). Drop
the threshold to **0.1** and it misses only **1** — at the cost of **14 benign patients flagged** for a
follow-up they did not need. Neither is "correct" in the abstract: the right cut depends on the clinical
cost ratio. What makes the choice honest rather than a guess is that the probability behind it is
calibrated.

## Reading the model

Unlike k-NN or naive Bayes, logistic regression is **interpretable**: each standardized coefficient is
that measurement's contribution to the malignant-vs-benign log-odds. And **L1** (`l1_ratio=1`, `saga`)
can prune the 30 features to a handful — automatic selection that shows *which* measurements carry the
signal.

In [ ]:
coef = logreg[-1].coef_.ravel()
feat = X.columns.to_numpy()
order = np.argsort(-np.abs(coef))[:8]

n_nonzero = []
for C in [0.02, 0.2, 1.0]:
    pipe = make_pipeline(
        StandardScaler(), LogisticRegression(l1_ratio=1.0, solver="saga", C=C, max_iter=200000)
    )
    pipe.fit(Xtr, ytr)
    n_nonzero.append(int(np.sum(np.abs(pipe[-1].coef_) > 1e-6)))
print(f"L1 nonzero of 30: C=0.02->{n_nonzero[0]}, C=0.2->{n_nonzero[1]}, C=1.0->{n_nonzero[2]}")

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [2, 1]})
rows = np.arange(len(order))[::-1]
bar_colors = [colors.COLORS["error"] if c > 0 else colors.COLORS["model"] for c in coef[order]]
axL.barh(rows, coef[order], color=bar_colors)
axL.set_yticks(rows)
axL.set_yticklabels([feat[i] for i in order], fontsize=8)
axL.axvline(0, color=colors.COLORS["text"], linewidth=0.8)
axL.set_xlabel("coefficient  (positive -> malignant)")
axL.set_title("Which measurements drive the diagnosis")
axR.plot([0.02, 0.2, 1.0], n_nonzero, color=colors.COLORS["model"], linewidth=2, marker="o")
axR.set_xscale("log")
axR.set_xlabel("C  (L1 strength)")
axR.set_ylabel("features kept (of 30)")
axR.set_title("L1 selects")
plt.tight_layout()
plt.show()

**Read the figure.** *Left:* the strongest positive weights — radius error, worst radius, mean concave
points, worst symmetry — are all **size and irregularity** measurements: bigger, more concave, more
irregular tumours read malignant, which is clinically sensible (the lone blue bar, compactness error,
leans the other way). *Right:* as the L1 penalty strengthens
(smaller `C`), the model keeps fewer features — 14, then 3 of the 30 — selecting the most informative
automatically. This is a story k-NN and naive Bayes could not tell.

## Error analysis & honest limits

The cancers the model misses sit in the **overlap band**, where its probability is near one half — it is
appropriately *unsure* there, and accuracy alone hid those misses while **recall surfaced them**.

Logistic regression draws a **linear** boundary. Where the true separation is curved, it **underfits** —
which is exactly the door into **decision trees** (chapter 04, non-linear partitions) and, far later,
neural networks (a sigmoid neuron *is* a logistic unit, and NB 4's gradient descent *is* how they train).

**When to use it:** a roughly linear boundary, interpretable weights, trustworthy probabilities, a strong
fast baseline. **When not:** strongly non-linear structure — reach for trees or a kernel.

## Your turn

1. **Set a policy.** From the recall/precision-vs-threshold curve, pick a threshold that misses **at most
   one** cancer on the test set. What recall does it give, and how many benign patients does it flag?
2. **Read the model.** List the top five malignant-driving coefficients and say, in plain words, what each
   one measures about the tumour.
3. **Recalibrate (harder).** Wrap the logistic `Pipeline` in `CalibratedClassifierCV` — it refits under
   internal cross-validation (try `isotonic`
   and `sigmoid`). Does the Brier score on the test set improve? By how much?

## What you built

A full honest workflow on real diagnostic data:

- model choice by **cross-validation on train**, then one **sealed test** (no leakage);
- **calibration** checked against naive Bayes — logistic regression's probabilities are far more
  trustworthy (the chapter-02 loop, closed);
- a **threshold chosen under asymmetric cost** — trading false alarms to miss fewer cancers;
- a **readable, L1-selectable** model whose coefficients tell a clinically sensible story.

**Vocabulary.** *calibration* · *reliability diagram* · *Brier score* · *decision threshold* ·
*false negative / recall* · *feature selection (L1)* · *cost asymmetry*.

## Chapter wrap — logistic regression, end to end

Six notebooks ago we had a straight line that produced an unbounded score. We turned it into a
probability (NB 1), read its geometry (NB 2), measured how wrong it was (NB 3), found the best weights by
rolling downhill (NB 4), drove the real estimator and its knobs (NB 5), and here put it to honest work on
a diagnosis that matters (NB 6).

Logistic regression is the course's first **discriminative** method and the first trained by **iterative
optimization** — the engine the back half of the course runs on. It is a strong, interpretable,
well-calibrated **linear** baseline. When the truth is not linear, we need a model that can carve the
space into pieces.

**Next: Module 04 — Decision Trees.**

## References

- Cox, D. R. (1958). *The regression analysis of binary sequences.* J. R. Stat. Soc. B, 20(2), 215–242.
  DOI: 10.1111/j.2517-6161.1958.tb00292.x
- Niculescu-Mizil, A., & Caruana, R. (2005). *Predicting good probabilities with supervised learning.*
  ICML. DOI: 10.1145/1102351.1102430
- Street, W. N., Wolberg, W. H., & Mangasarian, O. L. (1993). *Nuclear feature extraction for breast
  tumor diagnosis.* Proc. SPIE 1905. DOI: 10.1117/12.148698
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (§4.4).
  DOI: 10.1007/978-0-387-84858-7

---

*Previous: 05 — The estimator & its parameters*  ·  *Next: Module 04 — Decision Trees*